# ABSA — đánh giá và chuẩn bị nộp bài

Notebook có output chạy thật từ train.csv. Chưa có test_blind.csv nên chưa tạo file nộp; bản này chưa phải bộ nộp hoàn chỉnh. Không có API sinh nhãn hoặc dữ liệu thêm do LLM tạo.

Chạy hai suite trong HUONG_DAN.md trước để tạo model validation cục bộ. Các báo cáo dùng split 42 và 2026 phải đọc riêng. Checkpoint và ngưỡng được chọn trên tune; validation đã được xem trong quá trình phát triển nên điểm mang tính thăm dò.


In [1]:
import os, sys, json, time, subprocess
from pathlib import Path
import numpy as np, pandas as pd, joblib
ROOT = Path.cwd()
sys.path.insert(0,str(ROOT))
from absa_core import LABELS, reference_splits, prepare_texts, binary_predictions
from score_absa import report, show
from submission_io import validate_submission
RESULTS = Path(os.environ.get('ABSA_RESULTS_DIR',str(ROOT/'private_runs')))
train = pd.read_csv(ROOT/'train.csv')
assert list(train.columns)==['id','text']+LABELS
fit,tune,valid=reference_splits(train)
print('Train only:',len(train),'| fit:',len(fit),'| tune:',len(tune),'| validation:',len(valid))
print('Không đọc blind trong bước đánh giá.')


Train only: 4688 | fit: 2812 | tune: 938 | validation: 938
Không đọc blind trong bước đánh giá.


In [2]:
for name in ['experiment_comparison.csv','experiment_comparison_reference.csv']:
    print(name)
    display(pd.read_csv(RESULTS/name)[['experiment','ABSA_F1','CI95_low','CI95_high']])


experiment_comparison.csv


,experiment,ABSA_F1,CI95_low,CI95_high
0,Ensemble_lan1,0.782002,0.766254,0.796525
1,NBSVM_lan2,0.779170,0.761827,0.794291
2,NBSVM_lan1,0.772328,0.756214,0.788813
3,TFIDF_lan11,0.770924,0.754407,0.787261
4,TFIDF_lan5,0.768180,0.752470,0.783558
5,TFIDF_lan7,0.765548,0.748464,0.780515
6,TFIDF_lan6,0.764014,0.747407,0.779451
7,TFIDF_lan12,0.763084,0.746713,0.777792
8,TFIDF_lan8,0.762571,0.745066,0.779843
9,Ridge_lan1,0.760856,0.743290,0.776651


experiment_comparison_reference.csv


,experiment,ABSA_F1,CI95_low,CI95_high
0,NBSVM_lan3,0.757043,0.740667,0.770977
1,Ensemble_lan2,0.753736,0.735542,0.768727
2,TFIDF_lan15,0.753507,0.735755,0.768931
3,TFIDF_lan16,0.753363,0.737207,0.767507
4,TFIDF_lan14,0.752119,0.735075,0.767732
5,TFIDF_lan13,0.750468,0.732767,0.765698


## Chạy lại suy luận validation bằng model đã fit

TFIDF_lan14 được chọn cho bản refit minh họa vì có điểm tune tốt nhất trong suite reference, không chọn theo điểm validation. Các model đánh giá chỉ fit outer80, không chứa validation.


In [3]:
folder=RESULTS/'TFIDF_lan14'
bundle=joblib.load(folder/'validation_model.joblib')
started=time.perf_counter()
texts=prepare_texts(train.text.iloc[valid],bundle['config']['normalization'])
scores=bundle['model'].decision_function(bundle['features'].transform(texts))
pred=binary_predictions(scores,bundle['thresholds'])
seconds=time.perf_counter()-started
gt=train.iloc[valid][['id']+LABELS].reset_index(drop=True)
sb=pd.DataFrame(pred,columns=LABELS);sb.insert(0,'id',gt.id)
metrics=report(gt,sb,n_boot=1000)
show(metrics)
expected=json.loads((folder/'metrics.json').read_text())
assert abs(metrics['ABSA_F1']-expected['ABSA_F1'])<1e-12
print('End-to-end validation inference seconds:',round(seconds,4))
print('Mục tiêu ABSA-F1 0.96 đã đạt:',metrics['ABSA_F1']>=.96)


Số mẫu: 938
ABSA-F1 (điểm xếp hạng) = 0.752   CI95% [0.734, 0.768]
F1 theo cực: khen 0.829 | chê 0.675 | trung tính (phụ) 0.000 | phát hiện khía cạnh (phụ) 0.881
Nhãn yếu nhất: gia_tri_che F1 = 0.589; độ lệch chuẩn F1 14 nhãn = 0.116
Precision TB = 0.756 | Recall TB = 0.757 | Accuracy TB = 0.920 (14 nhãn)
  khen: P 0.830 R 0.832 | chê: P 0.681 R 0.683
kappa TB = 0.698 | MCC TB = 0.701 | Balanced acc TB = 0.850
F1 vi mô: khía cạnh = 0.891 | (khía cạnh, cực) = 0.776 | khớp đúng cả đánh giá = 0.367
Mốc lớp đa số =   nan; cải thiện tương đối =   nan

Khía cạnh      Nhắc F1 | Khen F1     P     R     κ     n | Chê F1      P     R     κ     n | Trung tính F1  n
nhan_vien      0.932  | 0.912 0.925 0.899 0.816   495 | 0.678  0.650 0.709 0.633   110 | 0.000       0
sach_se        0.953  | 0.962 0.950 0.974 0.943   310 | 0.656  0.837 0.539 0.633    76 | 0.000       0
vi_tri         0.923  | 0.900 0.899 0.902 0.850   315 | 0.602  0.633 0.574 0.579    54 | 0.000       2
phong          0.881  | 0.79

In [4]:
# Báo cáo 5 lỗi chứa văn bản train chỉ lưu riêng, không hiển thị lên repo công khai.
errors=json.loads((folder/'private_validation_errors.json').read_text())[:5]
notes=[
    'Cần phân biệt khía cạnh sạch sẽ với phòng; từ phòng không tự tạo cảm xúc cho khía cạnh phòng.',
    'Đánh giá nhiều mệnh đề: cần đối chiếu phạm vi lời chê về đường đi và tiện nghi với quy ước gốc.',
    'Mùi thơm là biểu đạt gián tiếp liên quan sạch sẽ; mô hình từ khóa có thể bỏ sót.',
    'Cụm cần tốt hơn là gợi ý cải thiện; mô hình dễ gán từ tốt thành lời khen.',
    'Bf là viết tắt ít gặp của bữa sáng, có thể dẫn đến bỏ sót khía cạnh ăn uống.'
]
parts=[]
for error,note in zip(errors,notes):
    parts.append(json.dumps(error,ensure_ascii=False,indent=2)+'\nNhận xét: '+note)
(folder/'private_error_analysis.md').write_text('\n\n'.join(parts),encoding='utf-8')
print('Đã tạo báo cáo riêng cho',len(errors),'lỗi; không sửa nhãn hoặc dự đoán blind.')


Đã tạo báo cáo riêng cho 5 lỗi; không sửa nhãn hoặc dự đoán blind.


In [5]:
# Refit minh họa từ toàn bộ train; không coi đây là điểm held-out của model mới.
subprocess.run([sys.executable,str(ROOT/'refit_final.py'),'--train',str(ROOT/'train.csv'),'--experiment',str(folder)],check=True)


{
  "fit_rows": 4688,
  "train_sha256": "665b581d11c5cb59eb5b3e6e730660fdd28567c64392f2e288e375dd082bd95e",
  "seconds": 7.082918113999995,
  "python": "3.12.14 (main, Aug 25 2026, 14:00:49) [Clang 22.1.3 ]",
  "platform": "Linux-6.18.57-x86_64-with-glibc2.41",
  "model": "final_model.joblib",
  "test_file_read": false,
  "thresholds_source": "inner tune; no blind labels or outer validation optimization"
}


CompletedProcess(args=['/workspace/work/absa_env/bin/python', '/workspace/work/KPDLWEB/refit_final.py', '--train', '/workspace/work/KPDLWEB/train.csv', '--experiment', '/workspace/work/KPDLWEB/TFIDF_lan14'], returncode=0)

In [6]:
# Đây chỉ là kiểm tra schema của file mẫu toàn 0, không phải file kết quả.
print(validate_submission(ROOT/'submission_demo.csv',ROOT/'submission_demo.csv'))


{'valid': True, 'rows': 732, 'columns': 22, 'template_ids_match': True, 'note': 'Template check only; independently compare IDs to test_blind.csv before submission.'}


## Bước nộp chạy cục bộ khi có dữ liệu blind

Chỉ đặt `ABSA_LOCAL_TEST` nếu muốn chạy mô hình đã huấn luyện trên máy riêng. Không đưa văn bản blind vào LLM. Mặc định cell sau không đọc blind. File submission chỉ có id và 21 nhãn 0/1. Trước khi nộp phải có kết quả kiểm tra chính thức bằng score_absa.py, đồng thời notebook phải giữ output và bộ nộp phải kèm môi trường, hướng dẫn cùng model.


In [7]:
local_test=os.environ.get('ABSA_LOCAL_TEST')
if local_test:
    output=folder/'submission.csv'
    subprocess.run([sys.executable,str(ROOT/'predict_local.py'),'--model',str(folder/'final_model.joblib'),'--local-test',local_test,'--template',str(ROOT/'submission_demo.csv'),'--output',str(output)],check=True)
    subprocess.run([sys.executable,str(ROOT/'score_absa.py'),'--kiem-tra',str(output),local_test],check=True)
else:
    print('Chưa chạy suy luận blind; chưa tạo submission.csv để nộp.')


Chưa chạy suy luận blind; chưa tạo submission.csv để nộp.
